This notebook is all about taking the data analysed by Sage and checking for the non-prime knots that it fails to identify, and also attempting to disambiguate knots which couldn't be uniquely identified.

#### Imports, setup

In [ ]:
import ast
import csv
import GridPythonModule as gpm
import os
import re

from multiprocessing import Pool
from pathlib import Path


# Sets the working directory
# This should be set to the folder containing this file.
# This step is unnecessary if working from the browser-based Jupyter interface,
# just comment this out if so.
WORKING_DIRECTORY = ""
os.chdir(Path(WORKING_DIRECTORY))


# Set the working directory first to ensure that this module is found properly
from utils import get_knot_id, convert_to_sage

### Check current unidentified knots

In [ ]:
""" Current status
Here we examine the given file for knots which were not properly identified
during the initial search.
"""
# The file to examine
FILE = ""

with open(Path(FILE)) as csv_file:
    reader = csv.DictReader(csv_file)

    row_count = 0
    seen_ids = {}
    for line in reader:
        row_count += 1
        if not line['id'][0].isnumeric():
            if line['id'] not in seen_ids:
                seen_ids[line['id']] = [1, line['simplified_grid']]
            else:
                seen_ids[line['id']][0] += 1
    
    total = 0
    for key, value in seen_ids.items():
        print("id: ", key)
        print("grid: ", value[1])
        print("count: ", value[0])
        print()
        total += value[0]

print(f"out of {row_count} knots, {total} are unidentified ({float(total)/row_count*100:.2f}%)")

### Rerun identification

In [ ]:
""" Reidentifying
Here we attempt to identify knots that couldn't be identified during the first
search stage.
"""

# The directory from which to source files that will be cleaned.
# This directory must exist.
# Every file in the directory will be considered valid, with errors occuring if
# any are not csvs with the correct format.
SOURCE_DIRECTORY = "to-clean"

# The directory into which to save the cleaned files. They will be saved with
# the same name as the files in the source directory.
# This directory will be created if it does not already exist.
DESTINATION_DIRECTORY = "cleaned"

# The number of parallel processes to use
# On a CPU with hyperthreading or similar, using more processes than there are
# physical cores can be very beneficial. Otherwise, on systems with many cores,
# more processes generally help.
NUM_PROCESSES = 10

# Crossing number threshold
# If a grid has crossing number above this threshold, we attempt to reduce it by
# scrambling and simplifying.
# This cannot be set above 13, or we risk misidentifying knots.
UNIQUENESS_THRESHOLD = 13

# The effort to apply in scrambling the grid. Higher effort directly corresponds
# to more random moves being performed on the grid. 
# Options are 'very_low', 'low', 'medium', 'high', 'very_high'.
# 'medium' is the default.
# See the GridPythonModule documentation for details.
SCRAMBLE_EFFORT = 'medium'

# The effort to apply in simplifying the grid. Higher effort directly
# corresponds to more random moves being applied to the grid.
# Options are 'low', 'medium', 'high'.
# 'medium' is the default.
# See the GridPythonModule documentation for details.
SIMPLIFY_EFFORT = 'medium'

# The maximum number of cleaning iterations to perform.
NUM_ITERATIONS = 50

# The maximum permitted ratio of unidentified knots to total rows. If the ratio
# falls below this threshold, the program will end early.
IDENTIFICATION_THRESHOLD = 0.005

def reanalyse(line_in: dict) -> tuple[dict, bool]:
    """
    Takes a line from a csv file as a dict and reexamines the grid to attempt to
    identify the knot represented.

    Grids which have more crossings than UNIQUENESS_THRESHOLD are scrambled and
    then simplified in an attempt to reduce crossing number to a point where a
    unique identification is possible.

    Grids with at most UNIQUENESS_THRESHOLD crossings have their actual number
    of crossings compared with all the potential knots they could be, with any
    knots of greater crossing number eliminated; if a grid is found to have only
    one possible knot, it is taken to be that knot. This is dependent on Sage
    returning *all* possible knots when giving a matching list. Connected sums
    are never eliminated from the matching list since their minimum crossing
    numbers are generally not known.
    """
    # If a knot is already identified, skip
    if line_in['id'][0].isnumeric():
        return line, False

    line = line_in.copy()

    # Keep track of whether the line we're analysing has been identified.
    is_unidentified = True

    # Only do stuff to the non-identified ones
    grid_string = line['simplified_grid']
    grid = ast.literal_eval(grid_string)
    grid_crossing = gpm.crossing_number(grid)

    # If a grid has too many crossings, attempt to simplify it
    # We can't guarantee uniqueness of an identification above 13
    if line['id'] == '-' or grid_crossing > UNIQUENESS_THRESHOLD:
        new_grid = gpm.scramble_grid(grid, effort = SCRAMBLE_EFFORT)
        new_grid = gpm.simplify_grid(new_grid, effort = SIMPLIFY_EFFORT)
        new_crossing = gpm.crossing_number(new_grid)

        if new_crossing >= grid_crossing:
            print(f"fail {grid} {grid_crossing} -> {new_grid} {new_crossing}\n", end='')
        else:
            print(f"simp {grid} {grid_crossing} -> {new_grid} {new_crossing}\n", end='')
            # The new grid reduced in crossing number, so we'll save it and attempt to
            # identify again
            line['simplified_grid'] = str(new_grid)
            if new_crossing < UNIQUENESS_THRESHOLD and len(ast.literal_eval(line['id'])) == 1:
                return line, is_unidentified
            try:
                new_sage_info = convert_to_sage(new_grid).get_knotinfo()
                new_info = get_knot_id(str(new_sage_info))
                print(f"got a unique id! {line['id']} -> {new_info}\n", end='')
                line['id'] = str(new_info)
                is_unidentified = False

            # Sage failed to produce a unique identification
            # We attempt to obtain a list of matches
            except NotImplementedError:
                try:
                    new_sage_info = convert_to_sage(new_grid).get_knotinfo(unique = False)
                    new_info = [get_knot_id(str(option)) for option in new_sage_info]
                    if str(new_info) in line['id']:
                        print("no new id\n", end='')
                    else:
                        print(f"new id! {line['id']} -> {new_info}\n", end='')
                        current_ids = ast.literal_eval(line['id'])
                        for new_id in new_info:
                            if new_id not in current_ids:
                                current_ids.append(new_id)
                        line['id'] = str(current_ids)

                # Sage couldn't identify the knot at all; it must have crossing number greater
                # than 13.
                except NotImplementedError:
                    print('Failed to identify at all\n', end='')

            return line, is_unidentified

    else:
        # Our grid was below the threshold, so we know that whatever options we've
        # found are the only possible ones.
        knot_ids = ast.literal_eval(line['id'])

        # Only one option remains, so it must be the unique id.
        if len(knot_ids) == 1:
            print(f"confirmed single option: {line['id']}\n", end='')
            line['id'] = knot_ids[0]
            is_unidentified = False

        # We have more than one option, so we eliminate any whose crossing number is
        # greater than the actual number of crossings in the grid.
        else:
            # This iteration is not nice to think about but necessary to safely delete from
            # the list while iterating over it.
            for i, knot_id in reversed(list(enumerate(knot_ids))):
                up_to = knot_id.find('n')
                if up_to == -1:
                    up_to = knot_id.find('_')

                if grid_crossing < int(knot_id[:up_to]) and '#' not in knot_id:
                    del knot_ids[i]

            knot_ids = set(knot_ids)

            # Only update if something has actually changed
            if set(ast.literal_eval(line['id'])) != knot_ids:
                print(f"reduced options: {line['id']} -> {knot_ids}\n", end='')
                line['id'] = str(list(knot_ids))

    return line, is_unidentified


def clean_file(file_path: str):
    """
    Takes the name of a file in SOURCE_DIRECTORY and runs all its rows through
    `reanalyse` according to the parameters above regarding thresholds and
    number of iterations.
    """
    temp_dir = Path('temp')
    os.makedirs(temp_dir, exist_ok = True)
    file_a = temp_dir / (file_path[:-4] + 'a' + '.csv')
    file_b = temp_dir / (file_path[:-4] + 'b' + '.csv')

    # Do an initial count of the source file's unidentified grids and write the
    # data to a temp file.
    with open(Path(SOURCE_DIRECTORY) / file_path, 'r') as data_file:
        reader = csv.DictReader(data_file)

        unidentified_count = 0
        row_count = 0

        with open(file_a, 'w') as out_file:
            writer = csv.DictWriter(out_file, ('id', 'scrambled_grid', 'simplified_grid', 'which', 'index'))
            writer.writeheader()

            for line in reader:
                row_count += 1
                if not line['id'][0].isnumeric():
                    unidentified_count += 1
                
                writer.writerow(line)

        print("##", file_path)
        print(f"{row_count} total band attachments")
        print(f"{unidentified_count} were not identified ({float(unidentified_count)/row_count * 100:.2f}%)")
        print()

    # Clean, iterating as necessary (until either NUM_ITERATIONS is reached, or
    # the ratio of unidentified knots falls below IDENTIFICATION_THRESHOLD)
    i = 1
    while i <= NUM_ITERATIONS:
        print(f"Beginning cleaning iteration #{i}")

        with open(file_a, 'r') as in_file:
            reader = csv.DictReader(in_file)
            with Pool(NUM_PROCESSES) as pool:
                result = pool.map(reanalyse, (line for line in reader))

        with open(file_b, 'w') as out_file:
            writer = csv.DictWriter(out_file, ('id', 'scrambled_grid', 'simplified_grid', 'which', 'index'))
            writer.writeheader()

            unidentified_count = 0
            for line, is_unidentified in result:
                writer.writerow(line)
                unidentified_count += is_unidentified

        if unidentified_count / row_count < IDENTIFICATION_THRESHOLD:
            print(f"Met threshold with {float(unidentified_count/row_count)*100:.2f}% unidentified")
            print(f"Stopping after {i} iterations")
            break

        if i == NUM_ITERATIONS:
            print(f"Stopping after {NUM_ITERATIONS} iterations")
            break

        file_a, file_b = file_b, file_a
        i += 1

    os.makedirs(DESTINATION_DIRECTORY, exist_ok = True)
    out_path = Path(DESTINATION_DIRECTORY) / file_path

    # Write out the data to the intended destination file
    with open(file_b, 'r') as in_file:
        reader = csv.DictReader(in_file)

        with open(out_path, 'w') as out_file:
            writer = csv.DictWriter(out_file, ('id', 'scrambled_grid', 'simplified_grid', 'which', 'index'))
            writer.writeheader()

            unidentified_count = 0
            for line in reader:
                if not line['id'][0].isnumeric():
                    unidentified_count += 1
                writer.writerow(line)

    # Clean up the temp files
    os.remove(file_a)
    os.remove(file_b)

    print()
    print(f"Finished {i} iterations with {unidentified_count} ({float(unidentified_count)/row_count*100:.2f}%) unidentified.")
    return unidentified_count, row_count

# Safety check to ensure we don't do any accidentally invalid identifications
if UNIQUENESS_THRESHOLD > 13:
    raise ValueError("UNIQUENESS_THRESHOLD cannot be above 13.")

files_to_clean = os.listdir(Path(SOURCE_DIRECTORY))
results = {}
for file_path in files_to_clean:
    results[file_path] = clean_file(file_path)

print()
print(f"Finished cleaning {len(files_to_clean)} files")

for file_path, pair in results.items():
    print()
    print(f"## {file_path}")
    print(f"{pair[0]} of {pair[1]} ({float(pair[0])/pair[1]*100:.2f}%) remain unidentified")
